# Memory

A quick overview of two broad types of agent memory:

- Short-term. Implemented with checkpoints that persist a thread's graph state. Stored in memory and scoped to specific threads
- Long-term. Persist application defined-data outside the graph state (i.e. graph database, external knowledge store). Used for cross-thread memory like user preference, facts and shared knowledge.

# Short-Term Memory

The Langchain create_agent function returns an agent graph, but it is stateless. It does not have memory of previous .invoke() calls. Memory is added across invocations by saving the state in a checkpointer.

In [1]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# InMemorySaver imported from langgraph's memory module
from langgraph.checkpoint.memory import InMemorySaver

from dotenv import load_dotenv

In [2]:
load_dotenv()

True

#### Configuring the Agent W/ Memory

Initialize the agent with an InMemorySaver() as an input to the checkpointer argument

In [13]:
agent = create_agent(
    model='claude-haiku-4-5',
    checkpointer=InMemorySaver(),
    system_prompt="""You are a chef and recommend recipes."""
)

Initialize a config object (dict) with a configurable key that contains the message thread as a dictionary. Pass it to the agent in the .invoke() call.

In [14]:
question = HumanMessage(content="I like gold potatoes better than russet potatoes. Can you recommend a dinner recipe for me?")
config = {'configurable': {'thread_id': '1'}}

In [15]:
response1 = agent.invoke({'messages': [question]},
             config,
             )

In [17]:
response1

{'messages': [HumanMessage(content='I like gold potatoes better than russet potatoes. Can you recommend a dinner recipe for me?', additional_kwargs={}, response_metadata={}, id='e662acde-208c-4812-b68c-2c7925be77e7'),
  AIMessage(content="# Gold Potato Dinner Recipe\n\nGreat choice! Gold potatoes have a naturally buttery flavor and creamy texture that's perfect for many dishes. Here's a wonderful option:\n\n## Creamy Gold Potato and Herb Chicken\n\n**Why it works:** Gold potatoes shine in creamy dishes, and their subtle sweetness pairs beautifully with chicken.\n\n### Simple preparation:\n1. **Roast the potatoes** - Cut gold potatoes into chunks, toss with olive oil, rosemary, and garlic, roast at 425°F until golden (about 25 minutes)\n2. **Pan-sear chicken** - Season chicken breasts and cook until done, set aside\n3. **Make a light cream sauce** - Deglaze the pan with white wine or broth, add cream, Dijon mustard, and fresh thyme\n4. **Combine** - Add the roasted potatoes to the sauce

In [18]:
question2 = HumanMessage(content="What's another recipe?")

In [19]:
response2 = agent.invoke({'messages': [question2]},
             config,
             )

In [20]:
response2

{'messages': [HumanMessage(content='I like gold potatoes better than russet potatoes. Can you recommend a dinner recipe for me?', additional_kwargs={}, response_metadata={}, id='e662acde-208c-4812-b68c-2c7925be77e7'),
  AIMessage(content="# Gold Potato Dinner Recipe\n\nGreat choice! Gold potatoes have a naturally buttery flavor and creamy texture that's perfect for many dishes. Here's a wonderful option:\n\n## Creamy Gold Potato and Herb Chicken\n\n**Why it works:** Gold potatoes shine in creamy dishes, and their subtle sweetness pairs beautifully with chicken.\n\n### Simple preparation:\n1. **Roast the potatoes** - Cut gold potatoes into chunks, toss with olive oil, rosemary, and garlic, roast at 425°F until golden (about 25 minutes)\n2. **Pan-sear chicken** - Season chicken breasts and cook until done, set aside\n3. **Make a light cream sauce** - Deglaze the pan with white wine or broth, add cream, Dijon mustard, and fresh thyme\n4. **Combine** - Add the roasted potatoes to the sauce

#### Observations

The results of a previous call to the agent are persisted to subsequent calls so the context is preserved. InMemorySaver and config work together to make this possible. InMemorySaver saves the checkpoints of the agent graph in memory. They are indexed by the thread_id which serves as the primary key. Setting the thread_id is a critical implementation decision. Threads with the same thread_id will pass all previous messages to the LLM, consuming the context window and increasing token use.